# Notebook 01: Setup & Data Overview

**Phase 0 — Scaffolding & Data Pipeline**

This notebook validates the data pipeline end-to-end and serves as a reference
for the structure of the TPC-H dataset used throughout this project.

---

## Prerequisites

Before running this notebook:

```bash
# 1. Copy the env template and fill in credentials
cp .env.example .env

# 2. Start PostgreSQL
docker compose up -d

# 3. Generate Parquet files (~30–60 s)
uv run python scripts/generate_data.py

# 4. Seed PostgreSQL (~1–3 min)
uv run python scripts/seed_postgres.py
```

In [2]:
import pathlib
import sys

import duckdb
import pandas as pd

# Ensure the project root is on sys.path so ``config`` can be imported
# regardless of where JupyterLab was launched from.
# Walk up from cwd until we find pyproject.toml (works whether the kernel
# was started from the project root, the notebooks/ folder, or anywhere else).
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PARQUET_DIR = ROOT / "data" / "parquet"

# DuckDB in-memory connection for querying Parquet files directly.
duck = duckdb.connect()

print(f"DuckDB version : {duckdb.__version__}")
print(f"Parquet dir    : {PARQUET_DIR}")
print(f"Parquet exists : {PARQUET_DIR.exists()}")

DuckDB version : 1.5.0
Parquet dir    : b:\Projects\sql-and-duckdb-playbook\data\parquet
Parquet exists : True


---

## 1 · Why TPC-H?

TPC-H is the **industry-standard OLAP benchmark** published by the Transaction
Processing Performance Council.  It is chosen here for three reasons:

| Reason | Detail |
|:---|:---|
| **Reproducible** | Generated entirely in-process by DuckDB's `dbgen` extension — no download, no external dependency |
| **Principled** | Queries in the benchmarking phase (notebook 07) are anchored to the official TPC-H Q1/Q3/Q5 spec — not cherry-picked |
| **Realistic shape** | 8 normalised tables, 6 M+ rows in `lineitem`, real OLAP query patterns (aggregations, multi-table joins, date range filters) |

Scale factor 1 (`sf=1`) produces approximately **1 GB** of data — large enough
to make query plan differences observable, small enough to fit on a laptop.

---

## 2 · Schema Overview

```
  region (5)
    └─ nation (25)
         ├─ supplier (10 K)
         │     └─ partsupp (800 K) ──┐
         │                           │
         │    part (200 K) ──────────┤
         │                           │
         └─ customer (150 K)         │
               └─ orders (1.5 M)     │
                     └─ lineitem (6 M, references partsupp)
```

Row counts above are approximate for `sf=1`.  `lineitem` is the fact table —
almost every analytical query in this repo touches it.

---

## 3 · Row Counts

In [3]:
tables = [
    "region", "nation", "supplier", "part",
    "partsupp", "customer", "orders", "lineitem",
]

rows = []
for table in tables:
    parquet = PARQUET_DIR / f"{table}.parquet"
    count = duck.execute(
        f"SELECT COUNT(*) FROM read_parquet('{parquet}')"
    ).fetchone()[0]
    size_mb = parquet.stat().st_size / 1_048_576
    rows.append({"table": table, "rows": count, "parquet_size_mb": round(size_mb, 1)})

df_counts = pd.DataFrame(rows)
df_counts["rows"] = df_counts["rows"].map("{:,}".format)
df_counts.columns = ["Table", "Row Count", "Parquet Size (MB)"]
df_counts

,Table,Row Count,Parquet Size (MB)
0,region,5,0.0
1,nation,25,0.0
2,supplier,"10,000",0.8
3,part,"200,000",6.1
4,partsupp,"800,000",40.7
5,customer,"150,000",11.8
6,orders,"1,500,000",53.8
7,lineitem,"6,001,215",197.5


---

## 4 · Table Summaries via DuckDB `SUMMARIZE`

`SUMMARIZE` is a DuckDB-exclusive shortcut that computes per-column statistics
(count, nulls, min, max, mean, std) in a single pass — equivalent to running
`DESCRIBE` + multiple `SELECT` aggregates, but in one line.  It is one of the
DuckDB-specific features demonstrated in notebook 05.

In [4]:
# SUMMARIZE lineitem — the most important table in TPC-H.
lineitem_parquet = PARQUET_DIR / "lineitem.parquet"
duck.execute(f"SUMMARIZE SELECT * FROM read_parquet('{lineitem_parquet}')").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,l_orderkey,BIGINT,1,6000000,1279960,3000279.604204982,1732187.873480347,1502120,2990635,4499871,6001215,0.0
1,l_partkey,BIGINT,1,200000,186412,100017.98932999402,57735.690826505015,50028,99983,150036,6001215,0.0
2,l_suppkey,BIGINT,1,10000,8565,5000.602606138924,2886.9619987306187,2502,5003,7501,6001215,0.0
3,l_linenumber,BIGINT,1,7,7,3.0005757167506912,1.732431403651943,2,3,4,6001215,0.0
4,l_quantity,"DECIMAL(15,2)",1.00,50.00,57,25.507967136654827,14.426262537016896,13,26,38,6001215,0.0
5,l_extendedprice,"DECIMAL(15,2)",901.00,104949.50,1147468,38255.138484656854,23300.438710962106,18757,36713,55156,6001215,0.0
6,l_discount,"DECIMAL(15,2)",0.00,0.10,12,0.04999943011540163,0.031619855108125844,0,0,0,6001215,0.0
7,l_tax,"DECIMAL(15,2)",0.00,0.08,10,0.04001350893110812,0.025816551798842628,0,0,0,6001215,0.0
8,l_returnflag,VARCHAR,A,R,3,NaN,NaN,NaN,NaN,NaN,6001215,0.0
9,l_linestatus,VARCHAR,F,O,2,NaN,NaN,NaN,NaN,NaN,6001215,0.0


In [5]:
# SUMMARIZE orders — useful to see the date range and price distribution.
orders_parquet = PARQUET_DIR / "orders.parquet"
duck.execute(f"SUMMARIZE SELECT * FROM read_parquet('{orders_parquet}')").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,o_orderkey,BIGINT,1,6000000,1279960,2999991.5,1732051.3849205573,1510202,3004327,4504067,1500000,0.0
1,o_custkey,BIGINT,1,149999,100303,75006.04057466667,43304.48900767422,37496,75003,112517,1500000,0.0
2,o_orderstatus,VARCHAR,F,P,2,NaN,NaN,NaN,NaN,NaN,1500000,0.0
3,o_totalprice,"DECIMAL(15,2)",857.71,555285.16,1182929,151219.53763164,88621.43136365109,78035,144437,215480,1500000,0.0
4,o_orderdate,DATE,1992-01-01,1998-08-02,2321,1995-04-19 00:40:05.7216,NaN,1993-08-28,1995-04-21,1996-12-10,1500000,0.0
5,o_orderpriority,VARCHAR,1-URGENT,5-LOW,5,NaN,NaN,NaN,NaN,NaN,1500000,0.0
6,o_clerk,VARCHAR,Clerk#000000001,Clerk#000001000,860,NaN,NaN,NaN,NaN,NaN,1500000,0.0
7,o_shippriority,INTEGER,0,0,1,0.0,0.0,0,0,0,1500000,0.0
8,o_comment,VARCHAR,Tiresias about the quickly express ideas dete...,zzle? furiously ironic packages are sil,1428301,NaN,NaN,NaN,NaN,NaN,1500000,0.0


---

## 5 · Sample Data

In [6]:
# Five rows from lineitem — shows the granularity: one row per order line.
duck.execute(
    f"SELECT * FROM read_parquet('{lineitem_parquet}') LIMIT 5"
).df()

,l_orderkey,l_partkey,l_suppkey,l_linenumber,l_quantity,l_extendedprice,l_discount,l_tax,l_returnflag,l_linestatus,l_shipdate,l_commitdate,l_receiptdate,l_shipinstruct,l_shipmode,l_comment
0,1,155190,7706,1,17.0,21168.23,0.04,0.02,N,O,1996-03-13,1996-02-12,1996-03-22,DELIVER IN PERSON,TRUCK,to beans x-ray carefull
1,1,67310,7311,2,36.0,45983.16,0.09,0.06,N,O,1996-04-12,1996-02-28,1996-04-20,TAKE BACK RETURN,MAIL,according to the final foxes. qui
2,1,63700,3701,3,8.0,13309.60,0.10,0.02,N,O,1996-01-29,1996-03-05,1996-01-31,TAKE BACK RETURN,REG AIR,ourts cajole above the furiou
3,1,2132,4633,4,28.0,28955.64,0.09,0.06,N,O,1996-04-21,1996-03-30,1996-05-16,NONE,AIR,s cajole busily above t
4,1,24027,1534,5,24.0,22824.48,0.10,0.04,N,O,1996-03-30,1996-03-14,1996-04-01,NONE,FOB,"the regular, regular pa"


In [7]:
# A denormalised view: lineitem joined to orders and customer.
# This is the shape most window function queries operate on.
duck.execute(f"""
    SELECT
        o.o_orderkey,
        o.o_orderdate,
        c.c_name        AS customer,
        o.o_orderstatus AS status,
        l.l_linenumber  AS line,
        l.l_extendedprice * (1 - l.l_discount) AS net_price
    FROM read_parquet('{PARQUET_DIR}/lineitem.parquet')  l
    JOIN read_parquet('{PARQUET_DIR}/orders.parquet')    o
        ON l.l_orderkey = o.o_orderkey
    JOIN read_parquet('{PARQUET_DIR}/customer.parquet')  c
        ON o.o_custkey = c.c_custkey
    LIMIT 10
""").df()

,o_orderkey,o_orderdate,customer,status,line,net_price
0,1,1996-01-02,Customer#000036901,O,1,20321.5008
1,1,1996-01-02,Customer#000036901,O,2,41844.6756
2,1,1996-01-02,Customer#000036901,O,3,11978.6400
3,1,1996-01-02,Customer#000036901,O,4,26349.6324
4,1,1996-01-02,Customer#000036901,O,5,20542.0320
5,1,1996-01-02,Customer#000036901,O,6,46146.7488
6,2,1996-12-01,Customer#000078002,O,1,44694.4600
7,3,1993-10-14,Customer#000123314,F,1,50814.5670
8,3,1993-10-14,Customer#000123314,F,2,42116.8230
9,3,1993-10-14,Customer#000123314,F,3,37497.4272


---

## 6 · PostgreSQL Connectivity Check

In [8]:
import psycopg2
from config import settings

pg = psycopg2.connect(settings.dsn)

with pg.cursor() as cur:
    cur.execute("""
        SELECT
            schemaname,
            tablename,
            pg_size_pretty(pg_total_relation_size(schemaname || '.' || tablename)) AS size
        FROM pg_tables
        WHERE schemaname = 'public'
        ORDER BY pg_total_relation_size(schemaname || '.' || tablename) DESC
    """)
    pg_tables = cur.fetchall()

pg.close()

pd.DataFrame(pg_tables, columns=["Schema", "Table", "Total Size"])

,Schema,Table,Total Size
0,public,lineitem,1009 MB
1,public,orders,236 MB
2,public,partsupp,155 MB
3,public,part,37 MB
4,public,customer,31 MB
5,public,supplier,2056 kB
6,public,region,24 kB
7,public,nation,24 kB


---

## Summary

| ✅ | Item |
|:---|:---|
| ✅ | DuckDB `dbgen(sf=1)` generated 8 TPC-H tables |
| ✅ | All tables exported to `data/parquet/` (flat + year-partitioned orders) |
| ✅ | `sql/schema/tpch_tables.sql` DDL created all 8 PostgreSQL tables |
| ✅ | `seed_postgres.py` bulk-loaded all rows via psycopg2 `COPY FROM STDIN` |
| ✅ | PostgreSQL row counts and table sizes confirmed above |

**Next:** [Notebook 02 — Window Functions](02_window_functions.ipynb)